# TSP restringido — ILP (PuLP)

Formulación entera por posiciones de visita: $x_{i,p}=1$ si la ciudad $i$ ocupa la
posición $p\in\{1,2,3,4\}$ tras salir del origen. Se fija $x_{2,2}=1$ (ciudad 2 en
segunda posición visitada). Solver CBC vía PuLP.


In [ ]:
import itertools
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

import pulp


In [ ]:
nodes = [0, 1, 2, 3, 4]
origin = 0
d02 = 4
dist = {
    (0, 1): 3, (0, 2): d02, (0, 3): 4, (0, 4): 2,
    (1, 2): 3, (1, 3): 4, (2, 3): 1, (2, 4): 6,
}
for (i, j), value in list(dist.items()):
    dist[(j, i)] = value
BIG_M = 100

def edge_exists(i, j):
    return (i, j) in dist

def cost(i, j):
    if i == j:
        return BIG_M
    return dist.get((i, j), BIG_M)

def route_cost(route):
    return sum(cost(route[i], route[i + 1]) for i in range(len(route) - 1))

def is_valid_route(route):
    if route[0] != origin or route[-1] != origin:
        return False
    if sorted(route[1:-1]) != [1, 2, 3, 4]:
        return False
    return route[2] == 2

def generate_candidate_routes():
    candidates = []
    for perm in itertools.permutations([1, 2, 3, 4]):
        route = [origin] + list(perm) + [origin]
        if is_valid_route(route):
            candidates.append({
                "route": route,
                "cost": route_cost(route),
                "uses_invalid_edges": any(not edge_exists(route[i], route[i + 1]) for i in range(len(route) - 1)),
            })
    return candidates

candidates = generate_candidate_routes()
OPTIMAL = min((c for c in candidates if not c["uses_invalid_edges"]), key=lambda c: c["cost"])
OPTIMAL_COST = float(OPTIMAL["cost"])
print("Candidatas:", len(candidates), "| óptimo estricto:", OPTIMAL)

def resolve_metrics_dir() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        root = base / "Notebooks" / "Optimization" / "TSPOptimization"
        if root.is_dir():
            d = root / "metrics"
            d.mkdir(parents=True, exist_ok=True)
            return d
    d = Path.cwd().resolve() / "Notebooks" / "Optimization" / "TSPOptimization" / "metrics"
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_tsp_metrics(payload: dict, filename: str):
    path = resolve_metrics_dir() / filename
    path.write_text(json.dumps(payload, indent=2))
    print("Saved:", path)
    return path


## Modelo y resolución


In [ ]:
cities = [1, 2, 3, 4]
positions = [1, 2, 3, 4]

t0 = time.perf_counter()
prob = pulp.LpProblem("TSP_restricted", pulp.LpMinimize)
x = pulp.LpVariable.dicts("x", (cities, positions), cat="Binary")

# cada ciudad / posición una vez
for i in cities:
    prob += pulp.lpSum(x[i][p] for p in positions) == 1
for p in positions:
    prob += pulp.lpSum(x[i][p] for i in cities) == 1

# restricción: ciudad 2 en posición 2
prob += x[2][2] == 1

def city_at(p):
    return pulp.lpSum(i * x[i][p] for i in cities)

# coste: usamos arcos entre asignaciones (expansión bilineal -> ya lineal en productos x*x? 
# Para ILP clásico usamos variables de arco entre posiciones consecutivas)
# y_{i,j,p}: ciudad i en p y j en p+1
arcs = []  # (i,j,p) p=1..3 between visits; also origin links

# coste origen -> pos1
for i in cities:
    # term cost(0,i) * x[i][1]
    pass

obj = pulp.lpSum(cost(0, i) * x[i][1] for i in cities)
obj += pulp.lpSum(cost(i, 0) * x[i][4] for i in cities)
for p in [1, 2, 3]:
    for i in cities:
        for j in cities:
            if i == j:
                continue
            y = pulp.LpVariable(f"y_{i}_{j}_{p}", cat="Binary")
            # y <= x[i,p], y <= x[j,p+1], y >= x[i,p]+x[j,p+1]-1
            prob += y <= x[i][p]
            prob += y <= x[j][p + 1]
            prob += y >= x[i][p] + x[j][p + 1] - 1
            obj += cost(i, j) * y

prob += obj
status = prob.solve(pulp.PULP_CBC_CMD(msg=False))
elapsed = time.perf_counter() - t0

# decode
pos_city = {}
for p in positions:
    for i in cities:
        if pulp.value(x[i][p]) > 0.5:
            pos_city[p] = i
route = [0, pos_city[1], pos_city[2], pos_city[3], pos_city[4], 0]
sol_cost = route_cost(route)
feasible = is_valid_route(route) and all(edge_exists(route[i], route[i+1]) for i in range(len(route)-1))
success = feasible and abs(sol_cost - OPTIMAL_COST) < 1e-9
print("Status:", pulp.LpStatus[status])
print("Ruta:", route, "coste =", sol_cost, "éxito =", success)

payload = {
    "problem": "tsp",
    "solver": "ilp",
    "family": "classical",
    "protocol": {"n_runs": 1, "method": "pulp_cbc_position_assignment", "constraint": "x[2,2]=1"},
    "instance": {"nodes": nodes, "origin": origin, "d02": d02},
    "quality": {
        "optimal_cost": OPTIMAL_COST,
        "optimal_route": OPTIMAL["route"],
        "cost_mean": float(sol_cost),
        "cost_best": float(sol_cost),
        "gap_mean": float(sol_cost - OPTIMAL_COST),
    },
    "success_rate": float(success),
    "time_s": {"mean": float(elapsed), "std": 0.0, "total": float(elapsed)},
    "effort": {"milp_status": pulp.LpStatus[status], "n_binary_vars_positions": 16},
    "runs": [{"seed": 0, "cost": float(sol_cost), "route": route, "feasible": feasible, "success": bool(success), "time_s": float(elapsed)}],
}
save_tsp_metrics(payload, "ilp_tsp.json")
